# Fundamentals of Data Science  
## Week 7: API Request

---

## Table of Contents
1. Learning Objectives
2. Context & Motivation
3. Conceptual Foundation
4. Demonstration
5. Hands-on
6. Reflection
7. Deliverable Checklist

---

# 1) Learning Objectives

เมื่อจบคาบนี้ นักศึกษาจะสามารถ

1. อธิบายหลักการทำงานของ REST API และโครงสร้างข้อมูลแบบ JSON ได้
2. ดึงข้อมูลจาก API ด้วย Python และผสานเข้ากับ dataset หลักได้อย่างถูกต้อง
3. ประเมินข้อจำกัดของข้อมูลจาก API (เช่น rate limit, coverage, quality) ในบริบทงาน Data Science

# 2) Context & Motivation

## ทำไม Data Scientist ต้องใช้ API?

ในโปรเจค PM2.5 เราเริ่มจากข้อมูลคุณภาพอากาศที่มีอยู่แล้ว (historical PM2.5)
แต่ **คำถามเชิงวิเคราะห์ที่สำคัญ** เช่น

* “PM2.5 สูงขึ้นเพราะอะไร?”
* “สภาพอากาศมีบทบาทอย่างไร?”

ไม่สามารถตอบได้จาก PM2.5 เพียงชุดเดียว

**API** คือกลไกสำคัญที่ช่วยให้เรา:

* ดึงข้อมูลเสริม (เช่น อุณหภูมิ, ความชื้น, ความเร็วลม)
* เชื่อมโยงข้อมูลจากหลายแหล่งเข้าด้วยกัน
* ทำให้ dataset ใกล้เคียง “โลกจริง” มากขึ้น

> ถ้าไม่ใช้ API → การวิเคราะห์จะขาดบริบท และอาจสรุปผิด

---

# 3) Conceptual Foundation: API Basics

## 3.1 REST API คืออะไร

REST API คือรูปแบบการให้บริการข้อมูลผ่าน HTTP โดยใช้แนวคิด **resource-based**

ตัวอย่าง:

* `/weather`
* `/forecast`
* `/air-quality`

HTTP Method ที่พบบ่อย:

* `GET` – ขอข้อมูล
* `POST` – ส่งข้อมูล

---

## 3.2 JSON Structure

ข้อมูลจาก API มักอยู่ในรูป **JSON (JavaScript Object Notation)**

โครงสร้างทั่วไป:

```json
{
  "location": {...},
  "data": [...],
  "meta": {...}
}
```

ประเด็นสำคัญสำหรับ Data Scientist:

* nested structure (dict ซ้อน dict, list)
* ต้องเลือก field ที่ “มีความหมายเชิงวิเคราะห์”

---

## 3.3 Rate Limit & Reliability

ข้อจำกัดที่ต้องคำนึงถึง:

* จำกัดจำนวน request ต่อวัน / ต่อ minute
* ข้อมูลอาจหายบางช่วง
* API เปลี่ยน schema ได้ในอนาคต

---

# 4) Demonstration

In [ ]:
# ดึงข้อมูลสภาพอากาศจาก Weather API

import requests
import pandas as pd

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 13.7563,
	"longitude": 100.5018,
	"start_date": "2021-01-01",
	"end_date": "2024-12-31",
	"daily": ["temperature_2m_mean", "weather_code", "temperature_2m_max", "temperature_2m_min", "rain_sum", "wind_speed_10m_max", "wind_gusts_10m_max"],
	"timezone": "Asia/Bangkok",
}

response = requests.get(url, params=params)
data = response.json()

weather_df = pd.DataFrame({
  "date": data["daily"]["time"],
  "temp_max": data["daily"]["temperature_2m_max"],
  "temp_min": data["daily"]["temperature_2m_min"],
  "temp_mean": data["daily"]["temperature_2m_mean"],
  "weather_code": data["daily"]["weather_code"],
  "rain_sum": data["daily"]["rain_sum"],
  "wind_speed_10m_max": data["daily"]["wind_speed_10m_max"],
  "wind_gusts_10m_max": data["daily"]["wind_gusts_10m_max"]
})

weather_df.head()

,date,temp_max,temp_min,temp_mean,weather_code,rain_sum,wind_speed_10m_max,wind_gusts_10m_max
0,2021-01-01,26.6,17.5,22.2,3,0.0,9.7,21.2
1,2021-01-02,27.4,17.9,22.6,3,0.0,11.4,22.7
2,2021-01-03,29.3,19.1,24.2,3,0.0,11.5,23.8
3,2021-01-04,30.7,20.8,25.4,3,0.0,11.7,24.8
4,2021-01-05,30.2,22.0,26.4,3,0.0,10.8,18.4


### สิ่งที่ต้องสังเกต

* โครงสร้าง JSON → เลือกเฉพาะ `daily`
* แปลง list → DataFrame
* `date` คือ key สำคัญสำหรับการ merge

---

# 5) Hands-on Practice


## Task 1: Data Cleaning & Preparation

* แปลง `date` เป็น `datetime`
* ตรวจสอบ missing values

---

In [ ]:
import pandas as pd
weather_df['date'] = pd.to_datetime(weather_df['date'])
weather_df.isnull().sum()

,0
date,0
temp_max,0
temp_min,0
temp_mean,0
weather_code,0
rain_sum,0
wind_speed_10m_max,0
wind_gusts_10m_max,0


## Task 2: Integrate with PM2.5 Dataset

* โหลด PM2.5 dataset หลัก
* Merge ด้วย `date`
* ตรวจสอบ:

  * แถวที่ merge ไม่ติด
  * ช่วงเวลาที่ข้อมูลอากาศไม่มี

```python
merged_df = pm25_df.merge(weather_df, on="date", how="left")
```

---

In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

df_2021 = pd.read_csv("pm2.52021.csv")
df_2022 = pd.read_csv("pm2.52022.csv")
df_2023 = pd.read_csv("pm2.52023.csv")
df_2024 = pd.read_csv("pm2.52024.csv")

Saving pm2.52021.csv to pm2.52021 (2).csv
Saving pm2.52022.csv to pm2.52022 (2).csv
Saving pm2.52023.csv to pm2.52023 (2).csv
Saving pm2.52024.csv to pm2.52024 (2).csv


In [ ]:
for df in [df_2021, df_2022, df_2023]:
    df["date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y")

df_2024["date"] = pd.to_datetime(df_2024["Date"], format="%d/%m/%Y")

pm25_df = pd.concat(
    [df_2021, df_2022, df_2023, df_2024],
    ignore_index=True
).sort_values("date").reset_index(drop=True)

# merge
merged_df = pm25_df.merge(weather_df, on="date", how="left")
print("merged_df:", merged_df.shape)

# แถวที่ merge ไม่ติด
missing_mask = merged_df.isnull().any(axis=1)
missing_weather_rows = merged_df[missing_mask]

print("จำนวนแถวที่ merge ไม่ติด =", missing_weather_rows.shape[0])
print(missing_weather_rows.head())

# ช่วงเวลาที่ข้อมูลอากาศไม่มี
missing_dates = merged_df.loc[missing_mask, "date"]

print("ช่วงวันที่ที่ไม่มี weather")
print("Min date:", missing_dates.min())
print("Max date:", missing_dates.max())

print("\nวันที่ที่ไม่มีข้อมูล weather:")
print(missing_dates.unique())

merged_df: (1277, 107)
จำนวนแถวที่ merge ไม่ติด = 1277
       Date   02T   05T   10T   11T   12T   59T   61T   03T   50T  ...  111T  \
0  1/1/2021  27.0  20.0  22.0  25.0  22.0  20.0  25.0  24.0  25.0  ...   NaN   
1  1/2/2021  32.0  25.0  26.0  27.0  27.0  23.0  26.0  29.0  31.0  ...   NaN   
2  1/3/2021  46.0  37.0  33.0  41.0  40.0  38.0  29.0  44.0  44.0  ...   NaN   
3  1/4/2021  39.0  31.0  32.0  36.0  38.0  36.0  28.0  46.0  41.0  ...   NaN   
4  1/5/2021  50.0  31.0  31.0  32.0  44.0  28.0  24.0  67.0  41.0  ...   NaN   

   112T  105T  temp_max  temp_min  temp_mean  weather_code  rain_sum  \
0   NaN   NaN      26.6      17.5       22.2             3       0.0   
1   NaN   NaN      27.4      17.9       22.6             3       0.0   
2   NaN   NaN      29.3      19.1       24.2             3       0.0   
3   NaN   NaN      30.7      20.8       25.4             3       0.0   
4   NaN   NaN      30.2      22.0       26.4             3       0.0   

   wind_speed_10m_max  wind_gus

In [ ]:
merged_df.to_csv("Integrated Dataset v1 (PM2.5 + Weather).csv", index=False)

# 6) Reflection

ให้ผู้เรียนตอบเป็นข้อความใน Notebook

1. ตัวแปรสภาพอากาศใด “น่าจะ” ส่งผลต่อ PM2.5 มากที่สุด เพราะอะไร
2. ข้อมูลจาก API มีข้อจำกัดอะไรบ้างเมื่อเทียบกับข้อมูลภาคสนาม
3. หากข้อมูลอากาศหายบางวัน ควรจัดการอย่างไร (drop / impute / keep)

---

1. Wind speed เพราะ ถ้าลมแรง จะช่วยชะล้างและพัดกระจายฝุ่นในอากาศ ทำให้ค่า PM2.5 ลดลง
2. ความละเอียดของพื้นที่ต่ำ เพราะ API มักให้ข้อมูลระดับพื้นที่ของทั้งเมือง แต่ PM2.5 จริงๆอาจแตกต่างกันมากแต่ละเขตในเมือง / ความคลาดเคลื่อนของโมเดล เพราะ API บางตัวใช้การคำนวณหรือประมาณค่า เลยอาจจะไม่แม่นเท่าการวัดจริง
3. ขึ้นอยู่ตามจำนวนของข้อมูลที่หาย ถ้าหายน้อยสามารถ Impute ได้ เช่น interpolate แต่ถ้ามากก็ควร Drop เพราะหากนำไปวิเคราะห์ต่ออาจผิดพลาด

# 7) Deliverable Checklist

* [ ] ดึงข้อมูลจาก API สำเร็จ
* [ ] Integrated Dataset v1 (PM2.5 + Weather)
* [ ] ตอบ reflection ครบถ้วน

---